# 🎤 Q1 — Hindi ASR: Fine-tune Whisper-small & Evaluate on FLEURS
**Josh Talks | AI Researcher Intern — Speech & Audio**

---

### Pipeline Overview
| Step | Description |
|------|-------------|
| 1 | GPU Check + Install dependencies |
| 2 | Mount Google Drive & load dataset |
| 3 | Preprocess audio + clean transcriptions |
| 4 | Fine-tune `openai/whisper-small` |
| 5 | Evaluate baseline & fine-tuned model on FLEURS Hindi |
| 6 | Report WER table |

> ⚡ **Runtime**: Set to `T4 GPU` → Runtime → Change Runtime Type → GPU

## ✅ Step 0 — Check GPU & Install Dependencies

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected! Go to Runtime → Change Runtime Type → T4 GPU')

In [ ]:
%%capture
!pip install transformers datasets accelerate evaluate jiwer \
             librosa soundfile torchaudio huggingface_hub -q

## 📁 Step 1 — Mount Google Drive & Upload Dataset CSV

Upload your `dataset.csv` (provided by Josh Talks) to Google Drive.
The CSV must have these columns:
`user_id, recording_id, language, duration, rec_url_gcp, transcription_url, metadata_url`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── UPDATE THIS PATH to where your dataset.csv is ───
DATASET_CSV = '/content/drive/MyDrive/josh_talks/dataset.csv'
# ──────────────────────────────────────────────────────

## 🔧 Step 2 — Data Preprocessing

### What we do:
| Operation | Detail |
|-----------|--------|
| Download audio | From `rec_url_gcp` |
| Download transcriptions | From `transcription_url` |
| Duration filter | Keep 1s – 30s clips only |
| Resample | → 16 kHz mono (Whisper requirement) |
| Normalize amplitude | Peak-normalize to ±0.9 |
| Silence trimming | `librosa.effects.trim(top_db=30)` |
| Text cleaning | Strip control chars, normalize whitespace, remove non-Hindi punctuation |

In [ ]:
import os, re, json, requests
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm

RAW_AUDIO_DIR  = '/content/data/raw_audio'
PROC_AUDIO_DIR = '/content/data/processed_audio'
TRANS_DIR      = '/content/data/transcriptions'
TRAIN_MANIFEST = '/content/data/train_manifest.json'

TARGET_SR    = 16000
MAX_DURATION = 30
MIN_DURATION = 1

for d in [RAW_AUDIO_DIR, PROC_AUDIO_DIR, TRANS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Directories created ✅')

In [ ]:
def download_file(url, dest_path):
    if os.path.exists(dest_path):
        return True
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            f.write(r.content)
        return True
    except Exception as e:
        return False

def load_transcription(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
        try:
            data = json.loads(content)
            if isinstance(data, dict):
                return data.get('text', data.get('transcription', '')).strip()
            if isinstance(data, list):
                return ' '.join(d.get('text','') for d in data).strip()
        except:
            return content
    except:
        return ''

def clean_text(text):
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Keep Devanagari, Latin letters/digits, danda, apostrophe
    text = re.sub(r"[^\u0900-\u097F\u0020a-zA-Z0-9।']", '', text)
    text = text.lower()
    return text

def preprocess_audio(src, dst):
    try:
        audio, sr = librosa.load(src, sr=None, mono=True)
        if sr != TARGET_SR:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak * 0.9
        audio, _ = librosa.effects.trim(audio, top_db=30)
        sf.write(dst, audio, TARGET_SR, subtype='PCM_16')
        return len(audio) / TARGET_SR
    except:
        return None

print('Helper functions defined ✅')

In [ ]:
df_raw = pd.read_csv(DATASET_CSV)
print(f'Loaded {len(df_raw)} rows from dataset CSV')

records = []
stats = dict(total=len(df_raw), downloaded=0, skip_dur=0, skip_trans=0, skip_audio=0)

for _, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc='Preprocessing'):
    rec_id   = str(row['recording_id'])
    duration = float(row.get('duration', 0))

    if not (MIN_DURATION <= duration <= MAX_DURATION):
        stats['skip_dur'] += 1
        continue

    raw_path = os.path.join(RAW_AUDIO_DIR, f'{rec_id}.wav')
    if not download_file(str(row['rec_url_gcp']), raw_path):
        stats['skip_audio'] += 1
        continue

    trans_path = os.path.join(TRANS_DIR, f'{rec_id}.txt')
    if not download_file(str(row['transcription_url']), trans_path):
        stats['skip_trans'] += 1
        continue

    text = clean_text(load_transcription(trans_path))
    if not text:
        stats['skip_trans'] += 1
        continue

    proc_path = os.path.join(PROC_AUDIO_DIR, f'{rec_id}.wav')
    actual_dur = preprocess_audio(raw_path, proc_path)
    if actual_dur is None or not (MIN_DURATION <= actual_dur <= MAX_DURATION):
        stats['skip_dur'] += 1
        continue

    stats['downloaded'] += 1
    records.append({
        'recording_id':  rec_id,
        'audio_path':    proc_path,
        'transcription': text,
        'duration':      round(actual_dur, 2),
    })

df_proc = pd.DataFrame(records)

print(f'\n=== Preprocessing Stats ===')
for k, v in stats.items():
    print(f'  {k:20s}: {v}')
print(f'\nFinal samples : {len(df_proc)}')
print(f'Total duration: {df_proc["duration"].sum()/3600:.2f} hours')
print(f'Avg duration  : {df_proc["duration"].mean():.2f}s')

# Save manifests
df_proc.to_csv('/content/data/processed_manifest.csv', index=False)
with open(TRAIN_MANIFEST, 'w', encoding='utf-8') as f:
    for _, r in df_proc.iterrows():
        f.write(json.dumps({'audio': r['audio_path'], 'sentence': r['transcription'], 'id': r['recording_id']}, ensure_ascii=False) + '\n')

print('\nManifests saved ✅')

## 🤖 Step 3 — Fine-tune Whisper-small on Hindi Data

### Training Config
| Hyperparameter | Value |
|----------------|-------|
| Base model | `openai/whisper-small` |
| Language | Hindi |
| Learning rate | 1e-5 |
| Batch size | 16 (effective 32) |
| Warmup steps | 500 |
| Max steps | 4000 |
| FP16 | ✅ |
| Best model criterion | Lowest val WER |

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from datasets import Dataset, Audio as HFAudio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer,
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)

MODEL_NAME  = 'openai/whisper-small'
LANGUAGE    = 'Hindi'
TASK        = 'transcribe'
OUTPUT_DIR  = '/content/drive/MyDrive/josh_talks/whisper-small-hindi-finetuned'
SAMPLE_RATE = 16000

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer  = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor  = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

print('Processor loaded ✅')

In [ ]:
# Load processed manifest into HuggingFace Dataset
records = []
with open(TRAIN_MANIFEST, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line.strip()))

hf_ds = Dataset.from_list(records)
hf_ds = hf_ds.cast_column('audio', HFAudio(sampling_rate=SAMPLE_RATE))

# Train / Val split (95/5)
split    = hf_ds.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
val_ds   = split['test']
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate'], return_tensors='pt'
    ).input_features[0]
    batch['labels'] = tokenizer(batch['sentence']).input_ids
    return batch

print('Mapping features (this takes a few minutes)…')
train_ds = train_ds.map(prepare_dataset, remove_columns=train_ds.column_names, num_proc=2)
val_ds   = val_ds.map(prepare_dataset,   remove_columns=val_ds.column_names,   num_proc=2)
print('Feature extraction done ✅')

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

print('Data collator defined ✅')

In [ ]:
import torch

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE
model.generation_config.task     = TASK
model.generation_config.forced_decoder_ids = None

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy='steps',
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=['tensorboard'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print('Starting training… ⏳')
trainer.train()

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f'\nModel saved to → {OUTPUT_DIR} ✅')

## 📊 Step 4 — Evaluate on FLEURS Hindi Test Set

Evaluates **both** models on `google/fleurs hi_in test` split and prints WER.

In [ ]:
from datasets import load_dataset, Audio as HFAudio

print('Loading FLEURS Hindi test set from HuggingFace…')
fleurs_test = load_dataset('google/fleurs', 'hi_in', split='test', trust_remote_code=True)
fleurs_test = fleurs_test.cast_column('audio', HFAudio(sampling_rate=16000))
print(f'Loaded {len(fleurs_test)} test examples')

In [ ]:
from tqdm.notebook import tqdm

def transcribe_with_model(model_path, dataset, batch_size=8):
    """Run Whisper inference on a dataset, return (predictions, references)."""
    proc  = WhisperProcessor.from_pretrained(model_path, language=LANGUAGE, task=TASK)
    model = WhisperForConditionalGeneration.from_pretrained(model_path).to('cuda')
    model.eval()
    forced_ids = proc.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    preds, refs = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f'{model_path}'):
        batch = dataset.select(range(i, min(i+batch_size, len(dataset))))
        feats = proc(
            [s['array'] for s in batch['audio']],
            sampling_rate=16000, return_tensors='pt', padding=True
        ).input_features.to('cuda')
        with torch.no_grad():
            ids = model.generate(feats, forced_decoder_ids=forced_ids)
        preds.extend(proc.batch_decode(ids, skip_special_tokens=True))
        refs.extend(batch['transcription'])
    return preds, refs

print('Transcription function ready ✅')

In [ ]:
import evaluate
wer_metric = evaluate.load('wer')

print('\n🔵 Evaluating BASELINE (openai/whisper-small)…')
preds_base, refs = transcribe_with_model('openai/whisper-small', fleurs_test)
wer_base = 100 * wer_metric.compute(predictions=preds_base, references=refs)
print(f'Baseline WER: {wer_base:.2f}%')

pd.DataFrame({'reference': refs, 'prediction': preds_base}).to_csv(
    '/content/baseline_predictions.csv', index=False, encoding='utf-8')

In [ ]:
print('\n🟢 Evaluating FINE-TUNED model…')
FINETUNED_PATH = OUTPUT_DIR  # or update if loading from Drive
preds_ft, refs = transcribe_with_model(FINETUNED_PATH, fleurs_test)
wer_ft = 100 * wer_metric.compute(predictions=preds_ft, references=refs)
print(f'Fine-tuned WER: {wer_ft:.2f}%')

pd.DataFrame({'reference': refs, 'prediction': preds_ft}).to_csv(
    '/content/finetuned_predictions.csv', index=False, encoding='utf-8')

## 🏆 Step 5 — WER Results Table

In [ ]:
results = {
    'Whisper-small (Pretrained Baseline)':           wer_base,
    'Whisper-small (Fine-tuned on ~10h Hindi data)': wer_ft,
}

df_results = pd.DataFrame([
    {'Model': k, 'WER (%)': round(v, 2)} for k, v in results.items()
])

print('\n' + '='*60)
print(f'{"Model":<45} {"WER (%)":>10}')
print('='*60)
for _, row in df_results.iterrows():
    print(f'{row["Model"]:<45} {row["WER (%)"]:.2f}')
print('='*60)

improvement = wer_base - wer_ft
print(f'\nAbsolute improvement : {improvement:.2f}%')
print(f'Relative improvement : {improvement/wer_base*100:.1f}%')

df_results.to_csv('/content/drive/MyDrive/josh_talks/wer_results.csv', index=False)
print('\nSaved → wer_results.csv ✅')

# Pretty display
from IPython.display import display
display(df_results.style.format({'WER (%)': '{:.2f}'}).highlight_min(subset='WER (%)', color='lightgreen'))

## 📝 Preprocessing Methodology Summary

### Audio Preprocessing
1. **Download** — Audio files fetched from `rec_url_gcp` and cached locally
2. **Duration filter** — Clips outside [1s, 30s] removed (Whisper's context limit)
3. **Resampling** — All files converted to **16 kHz mono** using `librosa.resample()`
4. **Amplitude normalization** — Peak-normalized to ±0.9 to prevent clipping
5. **Silence trimming** — Leading/trailing silence removed via `librosa.effects.trim(top_db=30)`
6. **Format** — Saved as **16-bit PCM WAV** using `soundfile`

### Text Preprocessing
1. **Download** — Transcriptions fetched from `transcription_url` (supports plain text & JSON)
2. **Control char removal** — Strip non-printable characters
3. **Whitespace normalization** — Collapse multiple spaces
4. **Punctuation removal** — Keep only Devanagari Unicode range `\u0900–\u097F`, Latin alphanumerics, danda `।`
5. **Lowercase** — English code-switched words lowercased for consistency

### Why these choices?
- **16 kHz** is Whisper's native sampling rate — avoids internal resampling artifacts
- **30s limit** matches Whisper's fixed 30-second context window
- **Silence trimming** improves training efficiency and reduces WER on boundary tokens
- **Peak normalization** ensures consistent volume across recordings from different devices